
# Agentic AI Workflow and Tool Integration


In [1]:

import pandas as pd
import numpy as np
import joblib
from pathlib import Path

# Define project directories
DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = Path("../models")

# Load trained forecasting model
model_package = joblib.load(
    MODEL_DIR / "horizon_lightgbm.pkl"
)

forecast_model = model_package["model"]
feature_columns = model_package["feature_columns"]

# Load inventory recommendations
inventory_recommendations = pd.read_csv(
    PROCESSED_DIR / "inventory_recommendations.csv"
)

# Load business decision plan
business_decisions = pd.read_csv(
    PROCESSED_DIR / "business_decision_plan.csv"
)

# Verify
print("Agentic AI environment ready!")
print("Forecasting features:", len(feature_columns))
print("Inventory recommendations:", len(inventory_recommendations))
print("Business decisions:", len(business_decisions))

print(
    "Pending manager decisions:",
    (business_decisions["decision_status"] == "PENDING").sum()
)


Agentic AI environment ready!
Forecasting features: 10
Inventory recommendations: 273
Business decisions: 273
Pending manager decisions: 273



### 2. Build an Inventory Recommendation Lookup Tool


In [2]:

def get_inventory_recommendation(item_id):

    # Find the requested product
    result = inventory_recommendations.loc[
        inventory_recommendations["item_id"] == item_id
    ]

    # Handle products without a reorder recommendation
    if result.empty:
        return {
            "item_id": item_id,
            "status": "NOT_FOUND",
            "message": "No pending reorder recommendation found."
        }

    # Get the product's information
    product = result.iloc[0]

    return {
        "item_id": str(product["item_id"]),
        "priority": str(product["priority"]),
        "recommended_order_qty": int(
            product["recommended_order_qty"]
        ),
        "inventory_position": int(
            product["inventory_position"]
        ),
        "reorder_point": int(
            product["reorder_point"]
        ),
        "approval_status": str(
            product["approval_status"]
        )
    }


# Test our first tool
result = get_inventory_recommendation("FOODS_3_090")

print(result)


{'item_id': 'FOODS_3_090', 'priority': 'HIGH', 'recommended_order_qty': 744, 'inventory_position': 56, 'reorder_point': 339, 'approval_status': 'PENDING'}



### 3. Build a Business Risk Lookup Tool


In [3]:

def get_business_risk(item_id):

    # Find the product in our business decision plan
    result = business_decisions.loc[
        business_decisions["item_id"] == item_id
    ]

    if result.empty:
        return {
            "item_id": item_id,
            "status": "NOT_FOUND",
            "message": "No business decision record found."
        }

    product = result.iloc[0]

    return {
        "item_id": str(product["item_id"]),
        "estimated_order_cost": round(
            float(product["estimated_order_cost"]), 2
        ),
        "allocated_order_qty": int(
            product["new_allocated_qty"]
        ),
        "allocated_order_cost": round(
            float(product["new_allocated_cost"]), 2
        ),
        "shortage_severity": str(
            product["shortage_severity"]
        ),
        "management_action": str(
            product["management_action"]
        ),
        "decision_status": str(
            product["decision_status"]
        )
    }


# Test our business tool
result = get_business_risk("FOODS_3_090")

print(result)


{'item_id': 'FOODS_3_090', 'estimated_order_cost': 14560.08, 'allocated_order_qty': 0, 'allocated_order_cost': 0.0, 'shortage_severity': 'MODERATE', 'management_action': 'REVIEW_DELIVERY_SCHEDULE', 'decision_status': 'PENDING'}



### 4. Build a Coordinator for Inventory and Business Tools


In [4]:

def coordinate_product_analysis(item_id):

    # Ask the inventory tool
    inventory_info = get_inventory_recommendation(item_id)

    # Ask the business tool
    business_info = get_business_risk(item_id)

    # Check that both tools found the product
    if (
        inventory_info.get("status") == "NOT_FOUND"
        or business_info.get("status") == "NOT_FOUND"
    ):
        return {
            "item_id": item_id,
            "status": "INCOMPLETE_DATA",
            "message": "Product information is unavailable in one or more tools."
        }

    # Combine both results
    report = {
        "item_id": item_id,
        "inventory": inventory_info,
        "business": business_info,
        "overall_status": "AWAITING_MANAGER_REVIEW"
    }

    return report


# Test the coordinator
coordinator_result = coordinate_product_analysis(
    "FOODS_3_090"
)

from pprint import pprint
pprint(coordinator_result)


{'business': {'allocated_order_cost': 0.0,
              'allocated_order_qty': 0,
              'decision_status': 'PENDING',
              'estimated_order_cost': 14560.08,
              'item_id': 'FOODS_3_090',
              'management_action': 'REVIEW_DELIVERY_SCHEDULE',
              'shortage_severity': 'MODERATE'},
 'inventory': {'approval_status': 'PENDING',
               'inventory_position': 56,
               'item_id': 'FOODS_3_090',
               'priority': 'HIGH',
               'recommended_order_qty': 744,
               'reorder_point': 339},
 'item_id': 'FOODS_3_090',
 'overall_status': 'AWAITING_MANAGER_REVIEW'}



### 5. Detect Inventory and Budget Conflicts


In [5]:

def detect_business_conflicts(item_id):

    # Get the combined report
    report = coordinate_product_analysis(item_id)

    if report.get("status") == "INCOMPLETE_DATA":
        return report

    inventory_info = report["inventory"]
    business_info = report["business"]

    conflicts = []

    # Check whether an urgent product has no funding
    if (
        inventory_info["priority"] == "HIGH"
        and business_info["allocated_order_qty"] == 0
    ):
        conflicts.append("HIGH_PRIORITY_WITHOUT_FUNDING")

    # Check whether the order is only partially funded
    if (
        0 < business_info["allocated_order_qty"]
        < inventory_info["recommended_order_qty"]
    ):
        conflicts.append("PARTIAL_FUNDING")

    # Check for serious delivery risk
    if business_info["shortage_severity"] == "SEVERE":
        conflicts.append("SEVERE_SHORTAGE_RISK")

    return {
        "item_id": item_id,
        "conflicts": conflicts,
        "conflict_count": len(conflicts),
        "requires_manager_review": len(conflicts) > 0
    }


# Test conflict detection
conflict_result = detect_business_conflicts(
    "FOODS_3_090"
)

print(conflict_result)


{'item_id': 'FOODS_3_090', 'conflicts': ['HIGH_PRIORITY_WITHOUT_FUNDING'], 'conflict_count': 1, 'requires_manager_review': True}



### 6. Generate Management Actions from Business Conflicts


In [6]:

def suggest_management_actions(item_id):

    # Find conflicts using our existing tool
    result = detect_business_conflicts(item_id)

    if result.get("status") == "INCOMPLETE_DATA":
        return result

    # Define actions for each type of conflict
    action_rules = {
        "HIGH_PRIORITY_WITHOUT_FUNDING": (
            "Review budget allocation or explore supplier payment terms."
        ),
        "PARTIAL_FUNDING": (
            "Review whether the funded quantity provides enough coverage."
        ),
        "SEVERE_SHORTAGE_RISK": (
            "Check faster delivery options and existing incoming orders."
        )
    }

    # Match conflicts with management suggestions
    actions = [
        action_rules[conflict]
        for conflict in result["conflicts"]
        if conflict in action_rules
    ]

    if not actions:
        actions = [
            "Continue standard inventory monitoring and manager review."
        ]

    return {
        "item_id": item_id,
        "conflicts": result["conflicts"],
        "suggested_actions": actions,
        "approval_status": "PENDING"
    }


# Test our action suggestion tool
action_result = suggest_management_actions(
    "FOODS_3_090"
)

from pprint import pprint
pprint(action_result)


{'approval_status': 'PENDING',
 'conflicts': ['HIGH_PRIORITY_WITHOUT_FUNDING'],
 'item_id': 'FOODS_3_090',
 'suggested_actions': ['Review budget allocation or explore supplier payment '
                       'terms.']}



### 7. Generate a Complete Product Decision Report


In [7]:

def generate_product_decision_report(item_id):

    # Get inventory and financial information
    report = coordinate_product_analysis(item_id)

    if report.get("status") == "INCOMPLETE_DATA":
        return report

    # Get detected conflicts and suggested actions
    actions = suggest_management_actions(item_id)

    inventory = report["inventory"]
    business = report["business"]

    # Build one complete report
    return {
        "item_id": item_id,
        "priority": inventory["priority"],
        "recommended_order_qty": inventory["recommended_order_qty"],
        "estimated_order_cost": business["estimated_order_cost"],
        "allocated_order_qty": business["allocated_order_qty"],
        "shortage_severity": business["shortage_severity"],
        "conflicts": actions["conflicts"],
        "suggested_actions": actions["suggested_actions"],
        "decision_status": "PENDING"
    }


# Test our complete report
final_report = generate_product_decision_report(
    "FOODS_3_090"
)

from pprint import pprint
pprint(final_report)


{'allocated_order_qty': 0,
 'conflicts': ['HIGH_PRIORITY_WITHOUT_FUNDING'],
 'decision_status': 'PENDING',
 'estimated_order_cost': 14560.08,
 'item_id': 'FOODS_3_090',
 'priority': 'HIGH',
 'recommended_order_qty': 744,
 'shortage_severity': 'MODERATE',
 'suggested_actions': ['Review budget allocation or explore supplier payment '
                       'terms.']}



### 8. Register Business Tools for Agent Coordination


In [8]:

# Register our existing business tools
tool_registry = {
    "inventory_lookup": {
        "function": get_inventory_recommendation,
        "description": "Retrieve product inventory and reorder information."
    },
    "business_risk_lookup": {
        "function": get_business_risk,
        "description": "Retrieve product costs, budget allocation and risks."
    },
    "conflict_detection": {
        "function": detect_business_conflicts,
        "description": "Identify inventory and business conflicts."
    },
    "management_actions": {
        "function": suggest_management_actions,
        "description": "Suggest management actions for business conflicts."
    },
    "decision_report": {
        "function": generate_product_decision_report,
        "description": "Generate a complete product decision report."
    }
}

# Check registered tools
print("Total registered tools:", len(tool_registry))

for name, details in tool_registry.items():
    print(f"{name}: {details['description']}")

# Test calling a registered tool
test_result = tool_registry["inventory_lookup"]["function"](
    "FOODS_3_090"
)

print("\nTool test:")
print(test_result["item_id"])
print("Recommended units:", test_result["recommended_order_qty"])


Total registered tools: 5
inventory_lookup: Retrieve product inventory and reorder information.
business_risk_lookup: Retrieve product costs, budget allocation and risks.
conflict_detection: Identify inventory and business conflicts.
management_actions: Suggest management actions for business conflicts.
decision_report: Generate a complete product decision report.

Tool test:
FOODS_3_090
Recommended units: 744



### 9. Build a Rule-Based Tool Router


In [9]:

def route_business_question(question, item_id):

    # Convert question to lowercase
    question = question.lower()

    # Choose a tool based on keywords
    if any(word in question for word in [
        "stock", "inventory", "reorder", "quantity"
    ]):
        tool_name = "inventory_lookup"

    elif any(word in question for word in [
        "cost", "budget", "afford", "price"
    ]):
        tool_name = "business_risk_lookup"

    elif any(word in question for word in [
        "conflict", "problem", "issue"
    ]):
        tool_name = "conflict_detection"

    elif any(word in question for word in [
        "action", "suggest", "recommendation"
    ]):
        tool_name = "management_actions"

    else:
        tool_name = "decision_report"

    # Call the selected tool
    result = tool_registry[tool_name]["function"](item_id)

    return {
        "question": question,
        "selected_tool": tool_name,
        "result": result
    }


# Test different questions
questions = [
    "How much stock should we reorder?",
    "What is the purchasing cost?",
    "Are there any business conflicts?",
    "What actions should management consider?"
]

for question in questions:
    response = route_business_question(
        question, "FOODS_3_090"
    )

    print("Question:", question)
    print("Selected tool:", response["selected_tool"])
    print()


Question: How much stock should we reorder?
Selected tool: inventory_lookup

Question: What is the purchasing cost?
Selected tool: business_risk_lookup

Question: Are there any business conflicts?
Selected tool: conflict_detection

Question: What actions should management consider?
Selected tool: management_actions




### 10. Build a Multi-Tool Business Router


In [10]:

def route_multiple_tools(question, item_id):

    question_lower = question.lower()
    selected_tools = []

    # Check inventory-related words
    if any(word in question_lower for word in [
        "stock", "inventory", "reorder", "quantity"
    ]):
        selected_tools.append("inventory_lookup")

    # Check financial-related words
    if any(word in question_lower for word in [
        "cost", "budget", "afford", "price"
    ]):
        selected_tools.append("business_risk_lookup")

    # Check conflict-related words
    if any(word in question_lower for word in [
        "conflict", "problem", "issue"
    ]):
        selected_tools.append("conflict_detection")

    # Check management-action words
    if any(word in question_lower for word in [
        "action", "suggest", "recommendation"
    ]):
        selected_tools.append("management_actions")

    # If no keywords match, use the full report
    if not selected_tools:
        selected_tools.append("decision_report")

    # Call each selected tool
    results = {}

    for tool_name in selected_tools:
        results[tool_name] = (
            tool_registry[tool_name]["function"](item_id)
        )

    return {
        "question": question,
        "item_id": item_id,
        "selected_tools": selected_tools,
        "tool_results": results
    }


# Test a question needing two tools
response = route_multiple_tools(
    "How much stock should we reorder, and can we afford it?",
    "FOODS_3_090"
)

print("Selected tools:", response["selected_tools"])

for tool_name, result in response["tool_results"].items():
    print(f"\n{tool_name}:")
    print(result)


Selected tools: ['inventory_lookup', 'business_risk_lookup']

inventory_lookup:
{'item_id': 'FOODS_3_090', 'priority': 'HIGH', 'recommended_order_qty': 744, 'inventory_position': 56, 'reorder_point': 339, 'approval_status': 'PENDING'}

business_risk_lookup:
{'item_id': 'FOODS_3_090', 'estimated_order_cost': 14560.08, 'allocated_order_qty': 0, 'allocated_order_cost': 0.0, 'shortage_severity': 'MODERATE', 'management_action': 'REVIEW_DELIVERY_SCHEDULE', 'decision_status': 'PENDING'}



### 11. Generate Manager-Friendly Business Responses


In [11]:

def generate_manager_response(question, item_id):

    # Get information from relevant tools
    response = route_multiple_tools(question, item_id)

    results = response["tool_results"]
    messages = []

    # Explain inventory recommendations
    if "inventory_lookup" in results:
        info = results["inventory_lookup"]

        if info.get("status") == "NOT_FOUND":
            messages.append(info["message"])
        else:
            messages.append(
                f"Inventory: {info['item_id']} has "
                f"{info['inventory_position']} units in inventory position. "
                f"The proposed reorder is "
                f"{info['recommended_order_qty']} units "
                f"with {info['priority']} priority."
            )

    # Explain financial information
    if "business_risk_lookup" in results:
        info = results["business_risk_lookup"]

        if info.get("status") == "NOT_FOUND":
            messages.append(info["message"])
        else:
            messages.append(
                f"Budget: The full proposed order costs approximately "
                f"${info['estimated_order_cost']:,.2f}. "
                f"Funding was allocated for "
                f"{info['allocated_order_qty']} units."
            )

            if info["allocated_order_qty"] == 0:
                messages.append(
                    "No purchasing budget was allocated to this product. "
                    "Management should review funding before ordering."
                )

    # Handle tools that return other report formats
    for tool_name in response["selected_tools"]:
        if tool_name not in [
            "inventory_lookup",
            "business_risk_lookup"
        ]:
            messages.append(
                f"{tool_name}: {results[tool_name]}"
            )

    messages.append(
        "This is a simulated recommendation. "
        "Any purchasing decision requires manager approval."
    )

    return "\n\n".join(messages)


# Test the response
manager_answer = generate_manager_response(
    "How much stock should we reorder, and can we afford it?",
    "FOODS_3_090"
)

print(manager_answer)


Inventory: FOODS_3_090 has 56 units in inventory position. The proposed reorder is 744 units with HIGH priority.

Budget: The full proposed order costs approximately $14,560.08. Funding was allocated for 0 units.

No purchasing budget was allocated to this product. Management should review funding before ordering.

This is a simulated recommendation. Any purchasing decision requires manager approval.



### 12. Generate Structured Business Decision Responses


In [12]:

def generate_structured_response(item_id):

    # Get the complete business report
    report = generate_product_decision_report(item_id)

    if report.get("status") == "INCOMPLETE_DATA":
        return report

    # Organize the result into clear sections
    structured = {
        "product": {
            "item_id": report["item_id"],
            "priority": report["priority"]
        },

        "inventory": {
            "recommended_order_qty": report["recommended_order_qty"]
        },

        "finance": {
            "estimated_order_cost": report["estimated_order_cost"],
            "allocated_order_qty": report["allocated_order_qty"]
        },

        "risk": {
            "shortage_severity": report["shortage_severity"],
            "conflicts": report["conflicts"]
        },

        "management": {
            "suggested_actions": report["suggested_actions"],
            "decision_status": "PENDING"
        }
    }

    return structured


# Test structured response
structured_result = generate_structured_response(
    "FOODS_3_090"
)

from pprint import pprint
pprint(structured_result)


{'finance': {'allocated_order_qty': 0, 'estimated_order_cost': 14560.08},
 'inventory': {'recommended_order_qty': 744},
 'management': {'decision_status': 'PENDING',
                'suggested_actions': ['Review budget allocation or explore '
                                      'supplier payment terms.']},
 'product': {'item_id': 'FOODS_3_090', 'priority': 'HIGH'},
 'risk': {'conflicts': ['HIGH_PRIORITY_WITHOUT_FUNDING'],
          'shortage_severity': 'MODERATE'}}



### 13. Define Tools for LLM-Based Coordination


In [13]:

# Define tools using JSON-compatible descriptions
llm_tool_definitions = []

for name, details in tool_registry.items():
    llm_tool_definitions.append({
        "type": "function",
        "name": name,
        "description": details["description"],
        "parameters": {
            "type": "object",
            "properties": {
                "item_id": {
                    "type": "string",
                    "description": "The product ID to analyze."
                }
            },
            "required": ["item_id"],
            "additionalProperties": False
        },
        "strict": True
    })

# Verify tool definitions
print("Tools prepared for LLM:", len(llm_tool_definitions))

for tool in llm_tool_definitions:
    print(tool["name"])

print("\nFirst tool definition:")
from pprint import pprint
pprint(llm_tool_definitions[0])


Tools prepared for LLM: 5
inventory_lookup
business_risk_lookup
conflict_detection
management_actions
decision_report

First tool definition:
{'description': 'Retrieve product inventory and reorder information.',
 'name': 'inventory_lookup',
 'parameters': {'additionalProperties': False,
                'properties': {'item_id': {'description': 'The product ID to '
                                                          'analyze.',
                                           'type': 'string'}},
                'required': ['item_id'],
                'type': 'object'},
 'strict': True,
 'type': 'function'}



### 14. Check LLM SDK and API Configuration


In [14]:

import os
import importlib.util

# Check whether the OpenAI Python package is installed
sdk_installed = (
    importlib.util.find_spec("openai") is not None
)

# Check whether an API key is configured
# Never display the actual key
api_key_configured = bool(
    os.environ.get("OPENAI_API_KEY")
)

print("OpenAI SDK installed:", sdk_installed)
print("API key configured:", api_key_configured)
print("Registered business tools:", len(llm_tool_definitions))

if sdk_installed and api_key_configured:
    print("Environment ready for API integration.")
else:
    print("Additional setup needed before API integration.")


OpenAI SDK installed: False
API key configured: True
Registered business tools: 5
Additional setup needed before API integration.



### 15. Install and Verify the OpenAI Python SDK


In [15]:

%pip install -U openai


  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   ---- -----------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
selenium 4.34.2 requires typing_extensions~=4.14.0, but you have typing-extensions 4.16.0 which is incompatible.


In [16]:

import openai
import os

print("OpenAI SDK version:", openai.__version__)
print("API key configured:", bool(os.environ.get("OPENAI_API_KEY")))
print("SDK installation: PASSED")


OpenAI SDK version: 3.42.0
API key configured: True
SDK installation: PASSED



### 16. Initialize OpenAI API Client


In [17]:

import os
from openai import OpenAI

# Check that the API key exists
assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY is not configured."
)

# Initialize the OpenAI client
client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)

# Verify the client without making an API request
print("OpenAI client initialized!")
print("API key available: True")
print("Responses API available:",
      hasattr(client, "responses"))
print("Registered business tools:", len(llm_tool_definitions))


OpenAI client initialized!
API key available: True
Responses API available: True
Registered business tools: 5



### 17. Simulate LLM Tool Selection Locally


In [19]:
import json

question = "Can we afford to purchase product FOODS_3_090?"
item_id = "FOODS_3_090"

# Use our existing rule-based router
local_result = route_business_question(
    question,
    item_id
)

# Simulate the tool-call format
simulated_tool_call = {
    "name": local_result["selected_tool"],
    "arguments": json.dumps({
        "item_id": item_id
    })
}

print("Mode: LOCAL SIMULATION")
print("Selected tool:", simulated_tool_call["name"])
print("Arguments:", simulated_tool_call["arguments"])
print("Tool selection simulated: True")
print("API requests made: 0")


Mode: LOCAL SIMULATION
Selected tool: business_risk_lookup
Arguments: {"item_id": "FOODS_3_090"}
Tool selection simulated: True
API requests made: 0



### 18. Execute a Registered Business Tool Safely


In [20]:

# Execute a tool selected by our local router
def execute_business_tool(tool_call):

    tool_name = tool_call["name"]

    # Only allow registered tools
    if tool_name not in tool_registry:
        return {
            "status": "BLOCKED",
            "reason": "Tool is not registered."
        }

    # Read and validate tool arguments
    try:
        arguments = json.loads(tool_call["arguments"])
    except (json.JSONDecodeError, TypeError):
        return {
            "status": "BLOCKED",
            "reason": "Invalid tool arguments."
        }

    # Only accept one non-empty product ID
    if (
        not isinstance(arguments, dict)
        or set(arguments) != {"item_id"}
        or not isinstance(arguments["item_id"], str)
        or not arguments["item_id"].strip()
    ):
        return {
            "status": "BLOCKED",
            "reason": "Invalid product ID."
        }

    # Execute the registered read-only function
    result = tool_registry[tool_name]["function"](
        arguments["item_id"]
    )

    return {
        "status": "SUCCESS",
        "tool": tool_name,
        "result": result
    }


# Execute our simulated tool call
execution_result = execute_business_tool(
    simulated_tool_call
)

from pprint import pprint
pprint(execution_result)


{'result': {'allocated_order_cost': 0.0,
            'allocated_order_qty': 0,
            'decision_status': 'PENDING',
            'estimated_order_cost': 14560.08,
            'item_id': 'FOODS_3_090',
            'management_action': 'REVIEW_DELIVERY_SCHEDULE',
            'shortage_severity': 'MODERATE'},
 'status': 'SUCCESS',
 'tool': 'business_risk_lookup'}



### 19. Test Tool Execution Safety Controls


In [21]:

# Test 1: An unregistered tool
unknown_tool = {
    "name": "send_payment",
    "arguments": json.dumps({
        "item_id": "FOODS_3_090"
    })
}

# Test 2: Invalid JSON
invalid_json = {
    "name": "business_risk_lookup",
    "arguments": "{invalid_json}"
}

# Test 3: Unexpected extra argument
extra_argument = {
    "name": "business_risk_lookup",
    "arguments": json.dumps({
        "item_id": "FOODS_3_090",
        "approve_order": True
    })
}

# Test 4: Valid registered tool
valid_tool = {
    "name": "inventory_lookup",
    "arguments": json.dumps({
        "item_id": "FOODS_3_090"
    })
}

tests = {
    "Unknown tool": unknown_tool,
    "Invalid JSON": invalid_json,
    "Unexpected argument": extra_argument,
    "Valid tool": valid_tool
}

for test_name, tool_call in tests.items():
    result = execute_business_tool(tool_call)

    print(f"{test_name}: {result['status']}")

    if result["status"] == "BLOCKED":
        print("Reason:", result["reason"])

    print()

# Verify expected behavior
assert execute_business_tool(unknown_tool)["status"] == "BLOCKED"
assert execute_business_tool(invalid_json)["status"] == "BLOCKED"
assert execute_business_tool(extra_argument)["status"] == "BLOCKED"
assert execute_business_tool(valid_tool)["status"] == "SUCCESS"

print("All four safety tests: PASSED")


Unknown tool: BLOCKED
Reason: Tool is not registered.

Invalid JSON: BLOCKED
Reason: Invalid tool arguments.

Unexpected argument: BLOCKED
Reason: Invalid product ID.

Valid tool: SUCCESS

All four safety tests: PASSED



### 20. Build a Complete Local Business Agent Workflow


In [22]:

def run_local_agent_workflow(question, item_id):

    # Step 1: Select tools using our local router
    routing = route_multiple_tools(question, item_id)

    # Step 2: Execute each tool through our safe dispatcher
    tool_results = {}

    for tool_name in routing["selected_tools"]:

        tool_call = {
            "name": tool_name,
            "arguments": json.dumps({"item_id": item_id})
        }

        execution = execute_business_tool(tool_call)

        tool_results[tool_name] = execution

    # Step 3: Return a consolidated report
    return {
        "question": question,
        "item_id": item_id,
        "mode": "LOCAL_RULE_BASED",
        "selected_tools": routing["selected_tools"],
        "tool_results": tool_results,
        "approval_status": "PENDING",
        "purchase_executed": False
    }


# Test the workflow
workflow_result = run_local_agent_workflow(
    "How much stock should we reorder, and can we afford it?",
    "FOODS_3_090"
)

from pprint import pprint
pprint(workflow_result)


{'approval_status': 'PENDING',
 'item_id': 'FOODS_3_090',
 'mode': 'LOCAL_RULE_BASED',
 'purchase_executed': False,
 'question': 'How much stock should we reorder, and can we afford it?',
 'selected_tools': ['inventory_lookup', 'business_risk_lookup'],
 'tool_results': {'business_risk_lookup': {'result': {'allocated_order_cost': 0.0,
                                                      'allocated_order_qty': 0,
                                                      'decision_status': 'PENDING',
                                                      'estimated_order_cost': 14560.08,
                                                      'item_id': 'FOODS_3_090',
                                                      'management_action': 'REVIEW_DELIVERY_SCHEDULE',
                                                      'shortage_severity': 'MODERATE'},
                                           'status': 'SUCCESS',
                                           'tool': 'business_risk_lookup'},
 


### 21. Create a Local Workflow Execution Log


In [23]:

from datetime import datetime, timezone

# Collect the workflow execution details
workflow_log = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "item_id": workflow_result["item_id"],
    "question": workflow_result["question"],
    "mode": workflow_result["mode"],
    "tools_used": workflow_result["selected_tools"],
    "tool_count": len(workflow_result["selected_tools"]),
    "all_tools_successful": all(
        result["status"] == "SUCCESS"
        for result in workflow_result["tool_results"].values()
    ),
    "approval_status": workflow_result["approval_status"],
    "purchase_executed": workflow_result["purchase_executed"]
}

# Verify the execution log
print("Product:", workflow_log["item_id"])
print("Execution mode:", workflow_log["mode"])
print("Tools used:", workflow_log["tools_used"])
print("Tool count:", workflow_log["tool_count"])
print("All tools successful:", workflow_log["all_tools_successful"])
print("Approval status:", workflow_log["approval_status"])
print("Purchase executed:", workflow_log["purchase_executed"])


Product: FOODS_3_090
Execution mode: LOCAL_RULE_BASED
Tools used: ['inventory_lookup', 'business_risk_lookup']
Tool count: 2
All tools successful: True
Approval status: PENDING
Purchase executed: False



### 22. Save and Verify Workflow Execution Log


In [24]:

import json
from pathlib import Path

# Create a folder for workflow logs
log_dir = Path("../data/processed/agent_logs")
log_dir.mkdir(parents=True, exist_ok=True)

# Define the log file
log_path = log_dir / "workflow_demo_log.json"

# Save the workflow log
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(workflow_log, f, indent=4)

# Load it again to verify
with open(log_path, "r", encoding="utf-8") as f:
    saved_log = json.load(f)

# Verify the saved information
assert saved_log == workflow_log
assert saved_log["all_tools_successful"] is True
assert saved_log["purchase_executed"] is False

print("Workflow log saved successfully!")
print("File exists:", log_path.exists())
print("Log verification: PASSED")
print("Execution mode:", saved_log["mode"])
print("Purchase executed:", saved_log["purchase_executed"])


Workflow log saved successfully!
File exists: True
Log verification: PASSED
Execution mode: LOCAL_RULE_BASED
Purchase executed: False



## Conclusion

Developed a local tool-based business coordination workflow integrating inventory recommendations and financial risk analysis.

### Key Achievements
- Registered five business analysis tools.
- Implemented single-tool and multi-tool routing.
- Developed conflict detection and management action recommendations.
- Added tool validation and execution safety checks.
- Generated structured business decision reports.
- Successfully saved and verified workflow execution logs.
- Maintained human approval requirements for all purchasing decisions.

### Limitations
The workflow currently uses rule-based tool selection. LLM integration was prepared but could not be executed due to unavailable API credits. No actual purchases or supplier transactions were performed.

**Next:** Notebook 07 — End-to-End Testing and Evaluation.
